In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))  # This brings 'src' into the path

import yaml

config_file = yaml.safe_load(
    open("../config.yaml", "r")
)
import os

# # ==== GPU selection ====
from autocvd import autocvd

autocvd(num_gpus=1)
# # =======================

# numerics
import jax.numpy as jnp
import numpy as np

# jf1uids data structures
from jf1uids import SimulationConfig
from jf1uids import SimulationParams
from jf1uids.option_classes.simulation_config import (
    BACKWARDS,
    FORWARDS,
    HLL,
    HLLC,
    MINMOD,
    OSHER,
    PERIODIC_BOUNDARY,
    BoundarySettings,
    BoundarySettings1D,
)

# jf1uids setup functions
from jf1uids import get_helper_data
from jf1uids.fluid_equations.fluid import construct_primitive_state
from jf1uids import get_registered_variables
from jf1uids.option_classes.simulation_config import finalize_config

# turbulent ic setup
from jf1uids.initial_condition_generation.turb import create_turb_field

# main simulation function
from jf1uids import time_integration

# units
from jf1uids import CodeUnits
from astropy import units as u
import astropy.constants as c

import h5py

from scipy.ndimage import convolve
import random

In [ ]:
base_config = SimulationConfig(
    runtime_debugging=config_file["turbulent_sim"]["runtime_debug"],
    first_order_fallback=config_file["turbulent_sim"]["first_order_fb"],
    progress_bar=config_file["turbulent_sim"]["progress_bar"],
    dimensionality=2,
    num_ghost_cells=config_file["turbulent_sim"]["num_ghost_cells"],
    box_size=config_file["turbulent_sim"]["box_size"],
    num_cells=config_file["turbulent_sim"]["num_cells"],
    fixed_timestep=True,  # to compare intermidiate states from low_res to high_res we need the fixed timestep
    num_timesteps=100,
    differentiation_mode=BACKWARDS,
    riemann_solver=HLL,
    exact_end_time=True,
    return_snapshots=config_file["turbulent_sim"]["return_snapshots"],
    num_snapshots=100,
)

registered_variables = get_registered_variables(base_config)

config_high_res = base_config._replace(
    num_cells=config_file["turbulent_sim"]["num_cells"]
)
config_low_res = base_config._replace(
    num_cells=config_file["turbulent_sim"]["num_cells"]
    // config_file["data"]["upsample_factor"]
)

helper_data_high_res = get_helper_data(config_high_res)
helper_data_low_res = get_helper_data(config_low_res)

# setup the unit system
code_length = 3 * u.parsec
code_mass = 1 * u.M_sun
code_velocity = 100 * u.km / u.s
code_units = CodeUnits(code_length, code_mass, code_velocity)

# time domain
C_CFL = 0.4

# set the final time of the simulation
t_final = 1.0 * 1e4 * u.yr
t_end = t_final.to(code_units.code_time).value

# simulation settings
gamma = 5 / 3

# turbulence
wanted_rms = 50 * u.km / u.s
dt_max = 0.1

# set the simulation parameters
params = SimulationParams(
    C_cfl=C_CFL,
    dt_max=dt_max,
    gamma=gamma,
    t_end=t_end,
)

# homogeneous initial state
rho_0 = 2 * c.m_p / u.cm**3
p_0 = 3e4 * u.K / u.cm**3 * c.k_B

rho = (
    jnp.ones(
        (
            config_high_res.num_cells,
            config_high_res.num_cells,
        )
    )
    * rho_0.to(code_units.code_density).value
)

# turbulence parameters
turbulence_slope = config_file["turbulent_sim"]["turbulence_slope"]
kmin = config_file["turbulent_sim"]["kmin"]
kmax = config_file["turbulent_sim"]["kmax"]

i = 0
a = config_high_res.num_cells // 2 - 10
b = config_high_res.num_cells // 2 + 10
p = (
    jnp.ones(
        (
            config_high_res.num_cells,
            config_high_res.num_cells,
        )
    )
    * p_0.to(code_units.code_pressure).value
)


In [ ]:
def downaverage_state(state: jnp.ndarray, downsample_factor: int) -> jnp.ndarray:
    """
    Downaverages the spatial dimensions of a state array using block reshaping.

    This function is designed for a state array with the shape
    (NUM_VARS, H, W) and reduces it to (NUM_VARS, h, w) by
    averaging over non-overlapping blocks.

    Args:
        state: The input JAX array with shape (NUM_VARS, H, W).
        target_shape: A tuple (h, w) representing the desired output
                      spatial dimensions. H must be divisible by h, and W
                      must be divisible by w.

    Returns:
        The downaveraged JAX array with shape (NUM_VARS, h, w).
    """
    # 1. Get input and output dimensions
    num_vars, h_in, w_in = state.shape
    h_out, w_out = h_in // downsample_factor, w_in // downsample_factor

    # 2. Assert that the downsampling is possible (dimensions are divisible)
    if h_in % h_out != 0 or w_in % w_out != 0:
        raise ValueError(
            f"Input shape {(h_in, w_in)} is not divisible by target shape {(h_out, w_out)}"
        )

    # 3. Calculate the block size (or downsampling factor)
    h_factor = h_in // h_out
    w_factor = w_in // w_out

    # 4. Reshape to create blocks and then take the mean
    # Original shape: (V, H, W)
    # Reshape to: (V, h_out, h_factor, w_out, w_factor)
    # This groups the original grid into blocks.
    reshaped = state.reshape(num_vars, h_out, h_factor, w_out, w_factor)

    # Take the mean over the block axes (h_factor and w_factor).
    # The axes are 2 and 4 in the reshaped array.
    downaveraged = reshaped.mean(axis=(2, 4))

    return downaveraged


In [ ]:
slice = random.randint(0, config_high_res.num_cells)
u_x = create_turb_field(config_high_res.num_cells, 1, turbulence_slope, kmin, kmax)[:,:,slice]
u_y = create_turb_field(config_high_res.num_cells, 1, turbulence_slope, kmin, kmax)[:,:,slice]

# scale the turbulence to the desired rms velocity
rms_vel = jnp.sqrt(jnp.mean(u_x**2 + u_y**2))
if not jnp.isfinite(rms_vel) or rms_vel == 0.0:
    raise ValueError("Skipping iteration due to bad rms_vel:", rms_vel)
u_x = u_x / rms_vel * wanted_rms.to(code_units.code_velocity).value
u_y = u_y / rms_vel * wanted_rms.to(code_units.code_velocity).value
# construct primitive state
print(rho.shape, u_x.shape, u_y.shape, p.shape)
initial_state_high_res = construct_primitive_state(
    config=config_high_res,
    registered_variables=registered_variables,
    density=rho,
    velocity_x=u_x,
    velocity_y=u_y,
    gas_pressure=p,
)

config_high_res = finalize_config(config_high_res, initial_state_high_res.shape)
result_high_res = time_integration(
    initial_state_high_res,
    config_high_res,
    params,
    helper_data_high_res,
    registered_variables,
)

if np.all(result_high_res.states[-1] == 0):
    print("0 last state")
    #raise ValueError("nan states")

initial_state_low_res = downaverage_state(
    initial_state_high_res, downsample_factor=config_file["data"]["upsample_factor"]
)

config_low_res = finalize_config(config_low_res, initial_state_low_res.shape)
result_low_res = time_integration(
    initial_state_low_res,
    config_low_res,
    params,
    helper_data_low_res,
    registered_variables,
)


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
result_high_res.states[0].shape

In [ ]:
initial_frame = 80
num_frames = 11
interval = 2
fig, axes = plt.subplots(num_frames, 4, figsize = (12, num_frames * 3))

for i in range(num_frames):
    hr_state = result_high_res.states[i * interval + initial_frame]
    low_state = result_low_res.states[i * interval + initial_frame]
    axes[i][0].imshow(hr_state[0])
    axes[i][1].imshow(low_state[0])
    axes[i][2].imshow(hr_state[3])
    axes[i][3].imshow(low_state[3])